# Treino: detector de empilhadeira (forklift) — fine-tuning YOLO

Treina um detector customizado de **empilhadeira** por cima do YOLO pré-treinado.
O modelo COCO conhece `person` e `truck`, mas **não** conhece empilhadeira — isso
resolve isso. Roteiro completo em [`docs/FINE_TUNING.md`](https://github.com/AliceCullen-html/ReconhecimentoPy/blob/main/docs/FINE_TUNING.md).

**Fluxo:** dataset rotulado → treinar → avaliar → exportar `best.pt` → usar na POC.

> ⚠️ **Use GPU!** `Ambiente de execução` → `Alterar o tipo de ambiente` → **T4 GPU**.
> Treinar em CPU é inviável (horas). Só footage/imagens royalty-free ou com licença adequada.

In [ ]:
# 1) Instala e confirma a GPU
!pip install -q ultralytics roboflow
import torch
print('GPU disponível:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Ligue a GPU: Ambiente de execução → Alterar tipo → T4 GPU'

## 2) Conseguir um dataset de empilhadeira (rotulado, formato YOLO)

O jeito mais rápido é o **[Roboflow Universe](https://universe.roboflow.com/search?q=forklift)** (grátis):

1. Crie uma conta grátis e pegue sua **API key** em `Settings → API`.
2. Busque por **`forklift`**, abra um dataset com boa quantidade de imagens.
3. Clique em **Download Dataset** → formato **YOLOv11** → **show download code**.
4. Copie o trecho `Roboflow(...)` e cole na célula abaixo (substituindo o exemplo).

> Alternativas sem Roboflow: dataset **LOCO** (Logistics Objects in Context) ou
> rotular seus próprios frames (CVAT/Label Studio) e subir em formato YOLO.

In [ ]:
# 2) Baixa o dataset. COLE aqui o snippet do Roboflow (Download → YOLOv11 → show code).
#    Exemplo do formato (troque pelos SEUS valores):
from roboflow import Roboflow
rf = Roboflow(api_key="COLE_SUA_API_KEY")
project = rf.workspace("WORKSPACE").project("PROJECT-forklift")
dataset = project.version(1).download("yolov11")

DATA_YAML = dataset.location + "/data.yaml"
print('data.yaml em:', DATA_YAML)
!cat "$DATA_YAML"

In [ ]:
# 3) Treina em cima do pré-treinado (transfer learning)
from ultralytics import YOLO

model = YOLO('yolo11s.pt')          # começa dos pesos COCO (yolo11n = mais leve)
results = model.train(
    data=DATA_YAML,
    epochs=50,                      # suba p/ 80-100 se tiver tempo/dados
    imgsz=640,
    batch=16,
    patience=15,                    # early stopping
    name='forklift',
)
print('Melhores pesos:', 'runs/detect/forklift/weights/best.pt')

In [ ]:
# 4) Avalia (mAP por classe)
best = YOLO('runs/detect/forklift/weights/best.pt')
metrics = best.val(data=DATA_YAML)
print('mAP50:', round(float(metrics.box.map50), 3))
print('mAP50-95:', round(float(metrics.box.map), 3))

In [ ]:
# 5) Testa visualmente: suba uma imagem/vídeo com empilhadeira
from google.colab import files
up = files.upload()                 # escolha um arquivo
src = next(iter(up))
best.predict(src, conf=0.30, save=True)
print('Resultado salvo em runs/detect/predict/ — veja no painel de Arquivos (📁).')

In [ ]:
# 6) Baixa o modelo treinado para usar na POC
from google.colab import files
files.download('runs/detect/forklift/weights/best.pt')

## Usar o `best.pt` na POC

⚠️ **Atenção às classes.** Um modelo treinado só em empilhadeira detecta **apenas**
`forklift` — ele deixa de conhecer `person`/`truck` do COCO. A POC hoje usa os
índices COCO (`0=person`, `7=truck`) para as flags da zona. Então há dois caminhos:

1. **Dataset combinado (recomendado):** treine com um dataset que tenha `person`,
   `truck` **e** `forklift` juntos. Aí é só apontar `model.weights` para o
   `best.pt` e ajustar `model.classes` no `config/config.yaml`.
2. **Dois modelos:** mantém o COCO para pessoa/caminhão e roda o `best.pt` só para
   empilhadeira, unindo as detecções. (Peça para eu adicionar esse suporte no
   `detector.py` + uma flag `forklift_in_zone`.)

Depois, para o gatilho de operação usar a empilhadeira, crie a flag em
`_compute_zone_flags` (`pipeline.py`) e `SUPPORTED_FLAGS` (`alerts.py`) — ver
`CLAUDE.md`. **Braço de carregamento** exige um dataset próprio (mais raro); mesmo
processo, com imagens rotuladas dessa classe.